In [ ]:
%%html
<style>
    h1 {color:purple}
    h2 {color:purple}
    h3 {color:#0099ff}
    hr {
        border: 0;
        height: 3px;
        background: #333;
        background-image: linear-gradient(to right, limegreen, deepskyblue, limegreen);
    }
</style>

# Outline
0. Part 0 — Intro/Setup
1. Part 1 — OpenAI Python APIs
2. **Part 2 — OpenAI Agents SDK**
   * 02-00. Agents SDK Introduction: building blocks, ReAct pattern
   * 02-01. Single Agent System — Python Tutor: `Agent`, `Runner.run()`, `RunResult`
   * 02-02. Conversation State in Agents: `previous_response_id`, `SQLiteSession`, `conversation_id`, `result.to_input_list()`
   * 02-03. Streaming Text and Events: `Runner.run_streamed()`, `StreamEvent`, `run_item_stream_event`
   * 02-04. Python Tutor with a Model-Backed Input Guardrail: `@input_guardrail`, `GuardrailFunctionOutput`, `InputGuardrailTripwireTriggered`
   * **02-05. Tools**
      * 02-05-00. Tools Overview: hosted, local, and custom tool taxonomy
      * 02-05-01. Financial Research Agent with a Custom Tool: `@function_tool`, hosted and custom tools
      * 02-05-02. Image Generation and Editing with `ImageGenerationTool`
      * 02-05-03. Multi-Agent Deitel Book Concierge: `FileSearchTool`, RAG, vector stores, handoffs
      * 02-05-04. Code Interpreter Tool: `CodeInterpreterTool`, hosted container
      * 02-05-05. Hosted MCP — Weather and Geocoding: `MCPServerStreamableHttp`, MCP authentication
      * 02-05-06. Local MCP — SQLite Database: `MCPServerStdio`
      * 02-05-07. AccuWeather Agent with `ComputerTool`: `AsyncComputer`, Playwright, `LocalBrowserComputer`
      * 02-05-08. `ShellTool` Folder Inspector: custom executor, human-in-the-loop approval
      * 02-05-09. **Local LLM via LiteLLM and Ollama: `LitellmModel`**
      * 02-05-10. Python Code Tutor with Dynamic Instructions: callable instructions, `RunContextWrapper`, typed context
3. Part 3 — Codex App
4. Part 4 — Wrap-Up and Additional References

---


# Creating Agents with the OpenAI Agents SDK
---

# Using a Local LLM via LiteLLM and Ollama
* Agents SDK programming model while changing the model provider
* Local models are useful for 
    * Privacy-sensitive drafts 
    * Offline experimentation
    * Cost saving
* Local model quality, speed and context limits vary significantly by model and hardware
    * Testing on my MacBook Pro M2 MAX with 96GB of RAM was SLOW
    * Tool use was flaky
    * Not as good at following instructions as OpenAI's models, even when using open-source "reasoning" models
    * For production workflows, test the exact model you plan to use with realistic inputs
* **LiteLLM** lets the Agents SDK call many model providers through one interface
* **`LitellmModel`** is a model adapter you can pass to `Agent(model=...)`
* **Ollama** runs open-weights models locally
* This example sends a local transcript file to a local model and asks for a concise summary
* **`set_tracing_disabled(True)`** avoids OpenAI tracing when running through a non-OpenAI provider

---

## Setup

* Install the Agents SDK LiteLLM extra:
    >```bash
    >pip install "openai-agents[litellm]"
    >```  
* Install Ollama from https://ollama.com/download  
* Pull a model you want to use:
    >```bash
    >ollama pull deepseek-r1:14b
    >```
* Run
    >```bash
    >ollama serve
    >```

---

## Imports and Constants

In [ ]:
from pathlib import Path

from IPython.display import Markdown, display
from agents import Agent, Runner, set_tracing_disabled
from agents.extensions.models.litellm_model import LitellmModel

set_tracing_disabled(disabled=True) # tracing works only with OpenAI hosted models

MODEL = 'ollama/deepseek-r1:14b' # model to run locally
APIKEY = 'ollama'  # Ollama ignores the key; LiteLLM requires a non-empty value.
API_BASE = 'http://localhost:11434'

---
## Load the Transcript

In [ ]:
TRANSCRIPT_PATH = Path('resources') / 'transcript.txt'
transcript = TRANSCRIPT_PATH.read_text(encoding='utf-8')

print(f'Loaded {TRANSCRIPT_PATH} ({len(transcript):,} characters)')

---
## Agent

* Same `Agent` and `Runner.run(...)` pattern as the OpenAI examples
* Model-specific piece is `LitellmModel(...)`


In [ ]:
summarizer_agent = Agent(
    name='Local Transcript Summarizer',
    model=LitellmModel(
        model=MODEL,
        api_key=APIKEY,
        base_url=API_BASE
    ),
    instructions="""
        You summarize technical presentation transcripts. Given a 
        transcript, create a summary abstract paragraph and key-points
        bullet list. Use straightforward sentences. Spell technical 
        features correctly. Avoid abbreviations. Do not refer to the 
        speaker. Do not invent details that are not present in the 
        transcript. If the transcript is unclear, say so. 
        
        DO NOT include original transcript in your response. 
    """
)

---
## Run the Summary Agent
* The transcript is passed as ordinary input text

In [ ]:
prompt = f"""
Summarize the following transcript.

Transcript:
{transcript}
"""

result = await Runner.run(summarizer_agent, prompt)
display(Markdown(result.final_output))

---
## References

* [Agents SDK LiteLLM models](https://openai.github.io/openai-agents-python/models/litellm/)
* [LiteLLM Ollama provider](https://docs.litellm.ai/docs/providers/ollama)
* [Ollama](https://ollama.com/)

---


© 2026 by Deitel & Associates, Inc. All Rights Reserved.